CAMBIO METODOLÓGICO IMPORTANTE respecto a versiones anteriores:

  A partir de ahora NO se reserva ningún holdout. Los 85 pacientes se usan
  íntegramente en la validación cruzada. La razón: con N=85 (y solo 17
  pacientes si se reservara un 20%), un holdout único tiene varianza muy
  alta y no aporta una estimación fiable adicional a la que ya da la CV
  repetida. El tutor indicó explícitamente: "si no [hay BD pública para
  holdout externo], lo dejo en CV". Se decidió no usar ninguna BD pública
  (no se encontró cohorte comparable en el plazo del TFM), así que la
  CV repetida pasa a ser la ÚNICA evaluación del trabajo.


Qué hace este script:
  1. Carga results/X_full.csv y y_full.csv (85 pacientes, 17 predictores), generados por el notebook 2
  2. Comprueba que X no contiene columnas no permitidas
  3. Evalúa XGBoost, Regresión Logística, Random Forest y TabPFN (si está
     disponible) con CV robusta 10-fold x 5 repeticiones (50 evaluaciones)
     sobre LOS 85 PACIENTES COMPLETOS. Se incluyen los CUATRO modelos para
     ser coherente con el notebook 06 (evaluación final), que compara los
     mismos cuatro.
  4. Entrena los pipelines FINALES sobre TODO el dataset (85 pacientes)
     y los guarda en models/ — estos son los modelos "de producción"
     del TFM, no hay un segundo conjunto de test que evaluarlos aparte:
     su evaluación de rendimiento es la CV del paso 3.

In [2]:
import os
import joblib
import pandas as pd
import numpy as np

In [3]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import xgboost as xgb

In [4]:
RUTA_DATASET = "data/dataset_modelado.csv"
CARPETA_MODELOS = "models"
CARPETA_RESULTADOS = "results"
SEMILLA = 42

In [5]:
os.makedirs(CARPETA_MODELOS, exist_ok=True)
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)

# 1. Carga de X e y (generadas en el notebook 2)

Se leen la matriz final y el target generados por el notebook 2 (results/X_full.csv y results/y_full.csv): 85 pacientes y 17 columnas ya codificadas. Los valores ausentes siguen sin imputar, porque se imputan dentro de cada fold de la validación cruzada. Requiere haber ejecutado antes el notebook 2.

In [7]:
X_encoded = pd.read_csv(f"{CARPETA_RESULTADOS}/X_full.csv")
y = pd.read_csv(f"{CARPETA_RESULTADOS}/y_full.csv").squeeze()

assert len(X_encoded) == len(y) == 85
assert 'ypN' not in X_encoded.columns and 'ypN_binario' not in X_encoded.columns \
       and 'Patient ID' not in X_encoded.columns, "ERROR: columna no permitida en X"

print("Shape de X:", X_encoded.shape, "| y:", y.shape)
print("Columnas:", list(X_encoded.columns))
print("\nDistribución de y (85 pacientes):")
print(y.value_counts(normalize=True).round(3))

Shape de X: (85, 17) | y: (85,)
Columnas: ['Age at diagnosis', 'Size of primary lesion', 'T', 'N', 'SUV max (at diagnosis)', 'Delta SUV max', 'Delta MTV', 'antraciclina', 'ciclos_antraciclina', 'platino', 'ciclos_platino', 'Tumor histology_ductal', 'HER2 (presence)_positive', 'ER (presence)_positive', 'Ki-67_low', 'Systemic therapy breast response (RECIST)_partial response', 'Systemic therapy axillar response (RECIST)_partial response']

Distribución de y (85 pacientes):
ypN_binario
0    0.647
1    0.353
Name: proportion, dtype: float64


In [12]:
# 3. Esquema de validación cruzada (10-fold x 5 repeticiones = 50 evaluaciones)
#90 % entrenamiento y 10 % test.
cv_robusta = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=SEMILLA)

Qué pasa en cada vuelta (10-fold)
Test: 8 o 9 pacientes (cinco folds de 9 y cinco de 8).
Entrenamiento: los 76 o 77 restantes.
Estratificado: cada fold conserva la proporción de ypN+ (35 %). Cada test tiene unos 3 ypN+ (30 entre 10 folds) y 5 o 6 ypN0. El entrenamiento tiene unos 27 ypN+ y 49 o 50 ypN0.

Se repite 5 veces con repartos distintos (n_repeats=5), así que hay 50 evaluaciones. El AUC que veremos, es la media de esos 50 AUC, y la desviación es entre folds.

In [13]:
# 4. Modelo 1: XGBoost Classifier (Pipeline con Imputador)


peso_clase_positiva = (y == 0).sum() / (y == 1).sum()
print(f"\nPeso aplicado a clase positiva en XGBoost (scale_pos_weight): {peso_clase_positiva:.2f}")

modelo_xgb = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3, 
    learning_rate=0.1, 
    scale_pos_weight=peso_clase_positiva,
    random_state=SEMILLA,
    eval_metric='logloss'
)

pipeline_xgb = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', modelo_xgb)
])

scores_xgb = cross_val_score(pipeline_xgb, X_encoded, y, cv=cv_robusta, scoring='roc_auc')
print(f"\n[XGBoost] AUC-ROC en CV (10-fold x 5 repeticiones, N=85):")
print(f"  -> Media: {scores_xgb.mean():.3f} (+/- {scores_xgb.std():.3f})")


Peso aplicado a clase positiva en XGBoost (scale_pos_weight): 1.83

[XGBoost] AUC-ROC en CV (10-fold x 5 repeticiones, N=85):
  -> Media: 0.820 (+/- 0.143)


In [14]:
# 5. Modelo 2: Regresión Logística Elastic Net

#C -> no se indica: vale 1.0 por defecto. Fuerza de la penalización. Un C pequeño penaliza más

pipeline_lr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), #valores ausentes con la mediana de entrenamiento de cada fold
    ('scaler', StandardScaler()), #cada variable con media 0 y desviacion 1
    ('classifier', LogisticRegression(
        penalty='elasticnet', #Combina dos penalizaciones que encogen los coeficientes: 
                              #L1 (puede llevarlos a exactamente 0) y L2 (los reduce de forma suave)
        l1_ratio=0.5,
        solver='saga', #algoritmo de optimización -> es el que admite la penalización elasticnet
        class_weight='balanced', #Da más peso a los errores en la clase minoritaria (ypN+)
        max_iter=5000, #alto para asegurar que el algoritmo converge.
        random_state=SEMILLA
    ))
])

scores_lr = cross_val_score(pipeline_lr, X_encoded, y, cv=cv_robusta, scoring='roc_auc')
print(f"\n[Regresión Logística ElasticNet] AUC-ROC en CV (10-fold x 5 repeticiones, N=85):")
print(f"  -> Media: {scores_lr.mean():.3f} (+/- {scores_lr.std():.3f})")


[Regresión Logística ElasticNet] AUC-ROC en CV (10-fold x 5 repeticiones, N=85):
  -> Media: 0.867 (+/- 0.136)


In [15]:
# 6. Modelo 3: Random Forest 
pipeline_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=4,
        class_weight='balanced',
        random_state=SEMILLA
    ))
])

#valores por defecto
#max_features='sqrt': en cada corte, cada árbol considera solo √17 ≈ 4 variables elegidas al azar.
#min_samples_split=2 y min_samples_leaf=1: casi no hay restricción en el tamaño de las ramas.
#bootstrap=True: cada árbol se entrena con una muestra de pacientes con reemplazo.

scores_rf = cross_val_score(pipeline_rf, X_encoded, y, cv=cv_robusta, scoring='roc_auc')
print(f"\n[Random Forest] AUC-ROC en CV (10-fold x 5 repeticiones, N=85):")
print(f"  -> Media: {scores_rf.mean():.3f} (+/- {scores_rf.std():.3f})")


[Random Forest] AUC-ROC en CV (10-fold x 5 repeticiones, N=85):
  -> Media: 0.851 (+/- 0.143)


In [16]:
#pip install tabpfn

In [17]:
#pip install --upgrade tabpfn

In [18]:
# 6. Modelo 3: TabPFN (Modelo Fundacional Tabular Exploratorio)

scores_tabpfn = None
try:
    from tabpfn import TabPFNClassifier

    pipeline_tabpfn = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('classifier', TabPFNClassifier(random_state=SEMILLA))
    ])

    scores_tabpfn = cross_val_score(pipeline_tabpfn, X_encoded, y, cv=cv_robusta, scoring='roc_auc')
    print(f"\n[TabPFN] AUC-ROC en CV (10-fold x 5 repeticiones, N=85):")
    print(f"  -> Media: {scores_tabpfn.mean():.3f} (+/- {scores_tabpfn.std():.3f})")
except Exception as e:
    print(f"\n[AVISO] No se pudo ejecutar TabPFN (omitiendo): {e}")


[TabPFN] AUC-ROC en CV (10-fold x 5 repeticiones, N=85):
  -> Media: 0.873 (+/- 0.136)


In [19]:
# 7. Entrenamiento FINAL sobre TODO el dataset (85 pacientes) y guardado
#    No hay holdout que reservar: estos son los modelos "de producción".
#    Su rendimiento reportado es el de la CV de los pasos 4-6, no una
#    evaluación adicional sobre datos no vistos (no existen, con N=85
#    todo el dataset se ha usado ya en la CV).

pipeline_xgb.fit(X_encoded, y)
pipeline_lr.fit(X_encoded, y)
pipeline_rf.fit(X_encoded, y)

joblib.dump(pipeline_xgb, f"{CARPETA_MODELOS}/xgboost_final_sin_holdout.pkl")
joblib.dump(pipeline_lr, f"{CARPETA_MODELOS}/logistic_regression_final_sin_holdout.pkl")
joblib.dump(pipeline_rf, f"{CARPETA_MODELOS}/random_forest_final_sin_holdout.pkl")

# Guardar resumen de resultados de CV
resumen = pd.DataFrame([
        {'Modelo': 'XGBoost', 'AUC_media': round(scores_xgb.mean(), 3), 'AUC_std': round(scores_xgb.std(), 3)},
        {'Modelo': 'Regresión Logística (ElasticNet)', 'AUC_media': round(scores_lr.mean(), 3), 'AUC_std': round(scores_lr.std(), 3)},
        {'Modelo': 'Random Forest', 'AUC_media': round(scores_rf.mean(), 3), 'AUC_std': round(scores_rf.std(), 3)},
    ])
if scores_tabpfn is not None:
    resumen = pd.concat([resumen, pd.DataFrame([{
        'Modelo': 'TabPFN', 'AUC_media': round(scores_tabpfn.mean(), 3), 'AUC_std': round(scores_tabpfn.std(), 3)
    }])], ignore_index=True)
resumen.to_csv(f"{CARPETA_RESULTADOS}/resumen_cv_baseline_sin_holdout.csv", index=False)

print("\n" + "=" * 60)
print("RESUMEN COMPARATIVO EN VALIDACIÓN CRUZADA (AUC-ROC, N=85, sin holdout)")
print("=" * 60)
print(f"  XGBoost Baseline:             {scores_xgb.mean():.3f}")
print(f"  Regresión Logística (SAGA):   {scores_lr.mean():.3f}")
print(f"  Random Forest:                {scores_rf.mean():.3f}")
if scores_tabpfn is not None:
    print(f"  TabPFN (Fundacional Tabular): {scores_tabpfn.mean():.3f}")
print("=" * 60)
print("\n[OK] Pipelines finales entrenados sobre los 85 pacientes y guardados en 'models/'.")
print("[OK] X_full.csv / y_full.csv guardados en 'results/' para los siguientes scripts.")
 


RESUMEN COMPARATIVO EN VALIDACIÓN CRUZADA (AUC-ROC, N=85, sin holdout)
  XGBoost Baseline:             0.820
  Regresión Logística (SAGA):   0.867
  Random Forest:                0.851
  TabPFN (Fundacional Tabular): 0.873

[OK] Pipelines finales entrenados sobre los 85 pacientes y guardados en 'models/'.
[OK] X_full.csv / y_full.csv guardados en 'results/' para los siguientes scripts.
